# 12 bis — Entraîner la perception par segmentation

Mini U-Net directement dans le notebook. Les classes et masques suivent le projet OUARTI initial. **Nécessite de vraies annotations de segmentation** (masques PNG 0..16, 255 pour inconnu), il ne génère jamais de faux labels.

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaEnv').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvre le depot IA_projet_Ouarti_ds_github dans VS Code : son dossier assetto_corsa_gym doit exister.')
PROJECT=REPO/'OUARTI'
if not PROJECT.is_dir():
    raise FileNotFoundError('Dossier OUARTI absent : extraire OUARTI a la racine du depot.')
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
print('Depot :',REPO)
print('Notebooks :',PROJECT)
print('Python :',sys.version.split()[0],sys.executable)


In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np,torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
import torch.nn.functional as F
CLASSES=['background','asphalt','white_line','yellow_line','curb','grass','gravel','soil',
         'runoff','vehicle','wall_barrier','tyres','brake_board','pit_entry_exit','checker','signal','other_obstacle']
IMAGES=PROJECT/'datasets'/'vision'/'images'
MASKS=PROJECT/'datasets'/'vision'/'masks'
images=sorted(IMAGES.glob('*.png'))
pairs=[(file,MASKS/file.name) for file in images if (MASKS/file.name).is_file()]
if len(pairs)<8:
    raise ValueError('Au moins 8 paires image + masque annote sont necessaires. '+
                     'Images dans datasets/vision/images; masques PNG meme nom dans datasets/vision/masks. '+
                     'Masque : classe 0 a 16, pixel inconnu 255.')
print(len(pairs),'paires correctement associees.')


In [ ]:
class SegDataset(Dataset):
    def __init__(self,pairs):self.pairs=pairs
    def __len__(self):return len(self.pairs)
    def __getitem__(self,idx):
        image_file,mask_file=self.pairs[idx]
        rgb=np.asarray(Image.open(image_file).convert('RGB'),dtype=np.float32)/255
        mask=np.asarray(Image.open(mask_file),dtype=np.uint8)
        if rgb.shape[:2]!=mask.shape:
            raise ValueError('Tailles image/masque differentes : '+str(image_file))
        if not np.isin(mask,list(range(17))+[255]).all():
            raise ValueError('Labels de masques invalides dans '+str(mask_file))
        return torch.from_numpy(rgb.copy()).permute(2,0,1),torch.from_numpy(mask.astype(np.int64))

class Block(nn.Sequential):
    def __init__(self,source,target):
        super().__init__(nn.Conv2d(source,target,3,padding=1,bias=False),nn.GroupNorm(4,target),nn.SiLU(),
                         nn.Conv2d(target,target,3,padding=1,groups=target,bias=False),
                         nn.Conv2d(target,target,1,bias=False),nn.GroupNorm(4,target),nn.SiLU())

class MiniUNet(nn.Module):
    def __init__(self,classes=17,width=16):
        super().__init__()
        self.down1=Block(3,width);self.down2=Block(width,width*2)
        self.bottom=Block(width*2,width*4)
        self.up2=Block(width*6,width*2);self.up1=Block(width*3,width)
        self.head=nn.Conv2d(width,classes,1)
    def forward(self,x):
        a=self.down1(x);b=self.down2(F.avg_pool2d(a,2));c=self.bottom(F.avg_pool2d(b,2))
        d=self.up2(torch.cat([F.interpolate(c,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        return self.head(self.up1(torch.cat([F.interpolate(d,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1)))


In [ ]:
# Separation par ordre de fichiers, pas de melange avant le split.
cut=max(1,int(.8*len(pairs)))
train_pairs=pairs[:cut];val_pairs=pairs[cut:]
if not val_pairs:raise ValueError('Pas assez de validations.')
train_loader=DataLoader(SegDataset(train_pairs),batch_size=2,shuffle=True,num_workers=0)
val_loader=DataLoader(SegDataset(val_pairs),batch_size=2,shuffle=False,num_workers=0)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net=MiniUNet().to(device)
optim=torch.optim.AdamW(net.parameters(),lr=2e-4)
EPOCHS=10
for epoch in range(EPOCHS):
    net.train();train_loss=[]
    for images,masks in train_loader:
        images,masks=images.to(device),masks.to(device)
        optim.zero_grad();logits=net(images)
        loss=F.cross_entropy(logits,masks,ignore_index=255)
        loss.backward();optim.step();train_loss.append(float(loss.item()))
    net.eval();valid=[]
    with torch.no_grad():
        for images,masks in val_loader:
            logits=net(images.to(device));valid.append(float(F.cross_entropy(logits,masks.to(device),ignore_index=255).item()))
    print('Epoch',epoch+1,'train',round(float(np.mean(train_loss)),5),'valid',round(float(np.mean(valid)),5))
checkpoint=PROJECT/'models'/'perception_miniunet.pt'
torch.save({'model':net.state_dict(),'classes':CLASSES,'width':16},checkpoint)
print('Modele :',checkpoint)
